In [20]:
import os
import joblib
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline , make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score,learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector

from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.metrics import confusion_matrix, classification_report

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression


import lightgbm as lgb  
import xgboost as xgb

from sklearn.calibration import CalibratedClassifierCV          #Calibrage de pbté

base_dir = os.path.dirname(os.getcwd())
print("Base Directory:", base_dir)
current_dir = os.getcwd()
print("Current Directory:", current_dir)

current_dir = os.getcwd()
base_dir = os.path.dirname(current_dir)
print("Base Directory:", base_dir)
data_dir = os.path.join(base_dir, 'data')
print("Data Directory:", data_dir)
model_dir = os.path.join(base_dir, 'models')
print("Model Directory:", model_dir)
results_dir = os.path.join(base_dir, 'results')
print("Results Directory:", results_dir)

Base Directory: /home/leon/dev/a_dev_bossGID/a_gid_M2/intero_churn/churn_client_prediction
Current Directory: /home/leon/dev/a_dev_bossGID/a_gid_M2/intero_churn/churn_client_prediction/notebook
Base Directory: /home/leon/dev/a_dev_bossGID/a_gid_M2/intero_churn/churn_client_prediction
Data Directory: /home/leon/dev/a_dev_bossGID/a_gid_M2/intero_churn/churn_client_prediction/data
Model Directory: /home/leon/dev/a_dev_bossGID/a_gid_M2/intero_churn/churn_client_prediction/models
Results Directory: /home/leon/dev/a_dev_bossGID/a_gid_M2/intero_churn/churn_client_prediction/results


In [21]:
data = pd.read_csv(base_dir +'/data/dataset_nettoye_final_500000.csv')
data

,Department,MonthlyIncome,Age,JobSatisfaction,TenureSatisfaction,EnvironmentSatisfaction,YearsAtCompany,OverTime,PromotionFrequency,Attrition
0,Research & Development,3910,38,3,18,4,6,1,0.142857,1
1,Sales,3639,33,2,14,4,7,0,0.125000,0
2,Sales,9549,40,1,4,2,4,0,0.200000,0
3,Technical,10153,47,3,24,3,8,0,0.555556,0
4,Research & Development,7367,33,2,0,3,0,0,3.000000,0
...,...,...,...,...,...,...,...,...,...,...
499995,Human Resources,9799,45,3,33,3,11,0,0.083333,0
499996,Research & Development,6605,34,3,12,2,4,0,0.600000,0
499997,Research & Development,5761,33,2,8,4,4,0,0.400000,0
499998,Research & Development,9507,28,4,24,4,6,1,0.428571,1


In [22]:
# Séparation des features et de la cible
X = data.iloc[:, :-1]
y = data.iloc[:, -1]

In [23]:
w = X['Department'].value_counts()
w

Department
Research & Development    250073
Sales                     149886
Technical                  50314
Human Resources            49727
Name: count, dtype: int64

In [24]:
# Identifier les colonnes quantitatives et qualitatives
num_features = make_column_selector(dtype_include=np.number)
cat_features = make_column_selector(dtype_exclude=np.number)


# Prétraitement pour les variables numériques (imputation + standardisation)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

# Prétraitement pour les variables catégorielles (imputation + encodage)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

# Application des transformations
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features)])

pipeline = Pipeline(steps=[('preprocessor', preprocessor),('classifier',xgb.XGBClassifier(n_estimators=1000,learning_rate=0.05,max_depth=8,subsample=0.8,colsample_bytree=0.8,random_state=42,n_jobs=-1))])


In [25]:
# Division des données

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entraînement et évaluation du modèle
pipeline.fit(X_train, y_train)

# Prédiction sur les données de test
y_pred = pipeline.predict(X_test)

# Évaluation des performances
accuracy = accuracy_score(y_test, y_pred)
print(f'La précision du modèle XGBClassifier est : {accuracy:.2f}')

scores = cross_val_score(pipeline, X_train, y_train, cv=5)
#print(f'Cross-validation scores: {scores.mean()}')
print(f'La précision moyenne en validation croisée est : {scores.mean():.2f}')


# Rapport de classification complet
print(classification_report(y_test, y_pred))

# Rapport de probabilité
y_prob = pipeline.predict_proba(X_test)[:, 1]  # Probabilité pour la classe 1 (fidel)

print(f"Probabilité d attrition : {y_prob[:5]}")  # Affiche les 5 premières probabilités


# Chemin du répertoire 'models' (assurez-vous que ce dossier existe déjà)
models_dir = os.path.join(base_dir, 'models')
os.makedirs(models_dir, exist_ok=True)  # si pas encore existant

# Sauvegarder le modèle avec joblib
model_path = os.path.join(models_dir, 'XGBClassifier_churn26.pkl')
joblib.dump(pipeline, model_path)  # Correction : utiliser directement model_path


La précision du modèle XGBClassifier est : 0.80
La précision moyenne en validation croisée est : 0.79
              precision    recall  f1-score   support

           0       0.80      0.99      0.89     79751
           1       0.44      0.04      0.07     20249

    accuracy                           0.80    100000
   macro avg       0.62      0.51      0.48    100000
weighted avg       0.73      0.80      0.72    100000

Probabilité d attrition : [0.18322761 0.21351948 0.03779048 0.16901153 0.07038266]


['/home/leon/dev/a_dev_bossGID/a_gid_M2/intero_churn/churn_client_prediction/models/XGBClassifier_churn26.pkl']